In [1]:
!pip install faker pandas pyarrow duckdb apache-airflow

In [2]:
import pandas as pd
import numpy as np
from faker import Faker
import uuid
import os

fake = Faker()

def generate_fake_data(output_path="./data"):
    os.makedirs(output_path, exist_ok=True)

    # 1. Eszközök létrehozása
    devices = []
    for _ in range(100):
        devices.append({
            "device_id": str(uuid.uuid4()),
            "model_name": fake.random_element(["Itron-G4", "Itron-G6", "SmartGas-2000"]),
            "location_id": fake.zipcode(),
            "city": fake.city()
        })
    devices_df = pd.DataFrame(devices)
    devices_df.to_csv(os.path.join(output_path, "devices.csv"), index=False)

    # 2. Fogyasztási adatok létrehozása
    consumption_data = []
    for _ in range(1000):
        dev = devices_df.sample(1).iloc[0]
        consumption_data.append({
            "reading_id": str(uuid.uuid4()),
            "device_id": dev["device_id"],
            "timestamp": fake.date_time_this_month(),
            "consumption": round(np.random.uniform(0.1, 5.0), 4),
            "status": fake.random_element(["OK", "LOW_BATTERY", "SIGNAL_LOSS"]),
            # A metaadatokat JSON-ként kezeljük a CSV-ben az egyszerűség kedvéért
            "temp": 20, 
            "pressure": 1.013
        })
    consumption_df = pd.DataFrame(consumption_data)
    # Figyelem: Itt a név consumption_data.csv lett a konzisztencia miatt
    consumption_df.to_csv(os.path.join(output_path, "consumption_data.csv"), index=False)

    print(f"Fake data generated and saved to {output_path}")
    print(f"Generated {len(consumption_df)} readings and {len(devices_df)} devices.")

if __name__ == "__main__":
    generate_fake_data()

Fake data generated and saved to ./data
Generated 1000 readings and 100 devices.


In [3]:
import pandas as pd
import duckdb
import os

def run_transformations(input_path="./data", db_path="./data/gas_warehouse.duckdb"):
    # Beolvasás (Javítva: .csv és a megfelelő fájlnév)
    df_readings = pd.read_csv(os.path.join(input_path, "consumption_data.csv"))
    df_devices = pd.read_csv(os.path.join(input_path, "devices.csv"))

    # 1. Adattisztítás
    df_readings = df_readings.dropna(subset=['consumption'])
    df_readings['timestamp'] = pd.to_datetime(df_readings['timestamp'])

    # 2. Adattranszformáció: Dim_Time létrehozása
    df_time = pd.DataFrame()
    df_time['timestamp'] = df_readings['timestamp'].unique()
    df_time['hour'] = df_time['timestamp'].dt.hour
    df_time['day'] = df_time['timestamp'].dt.day
    df_time['month'] = df_time['timestamp'].dt.month
    df_time['is_weekend'] = df_time['timestamp'].dt.dayofweek > 4

    # 3. Aggregáció
    # A dátum alapú aggregációhoz le kell vágni az időpontot
    df_readings_copy = df_readings.copy()
    df_readings_copy['date'] = df_readings_copy['timestamp'].dt.date
    df_daily_agg = df_readings_copy.groupby(['date', 'device_id'])['consumption'].sum().reset_index()
    df_daily_agg.columns = ['date', 'device_id', 'daily_total_consumption']

    # 4. Betöltés DuckDB-be
    con = duckdb.connect(db_path)

    # 5. Tábla létrehozása és adatok betöltése
    con.execute("CREATE OR REPLACE TABLE dim_device AS SELECT * FROM df_devices")
    con.execute("CREATE OR REPLACE TABLE dim_time AS SELECT * FROM df_time")
    con.execute("CREATE OR REPLACE TABLE fact_gas_usage AS SELECT * FROM df_readings")
    con.execute("CREATE OR REPLACE TABLE agg_daily_usage AS SELECT * FROM df_daily_agg")
    
    con.close()
    print(f"Transformation and loading to DuckDB finished at {db_path}.")

if __name__ == "__main__":
    run_transformations()

Transformation and loading to DuckDB finished at ./data/gas_warehouse.duckdb.


In [5]:
import sys
import os
from datetime import datetime
from airflow import DAG
from airflow.providers.standard.operators.python import PythonOperator

current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.append(current_dir)

default_args = {
    'owner': 'airflow',
    'start_date': datetime(2024, 1, 1),
    'retries': 1,
}

# A DAG definíciója
dag = DAG(
    'gas_iot_pipeline',
    default_args=default_args,
    schedule='@daily',
    catchup=False
)

t1 = PythonOperator(
    task_id='generate_data',
    python_callable=generate_fake_data, 
    op_kwargs={'output_path': './data'},
    dag=dag
)

t2 = PythonOperator(
    task_id='transform_data',
    python_callable=run_transformations,
    op_kwargs={'input_path': './data', 'db_path': './data/gas_warehouse.duckdb'},
    dag=dag
)

t1 >> t2

print("A DAG definíció sikeresen betöltve a memóriába.")

A DAG definíció sikeresen betöltve a memóriába.
